## Preparação dos dados de treinamento e teste

In [0]:
#Variaveis de Ambiente
try:
    catalog = dbutils.widgets.get("catalog")
except:
    catalog = "dev"
schema = "falha"

 # Define nomes completos das tabelas
train_table = f"{catalog}.{schema}.train_set"
test_table = f"{catalog}.{schema}.test_set"

In [0]:
import pandas as pd


In [0]:
# 2. Configuração do notebook
# Opção recomendada no Databricks: tabela Gold do Unity Catalog.
DATA_SOURCE = "datapulse.gold.sensor_events"

# Alternativa CSV: informe um caminho acessível ao driver do cluster,
# por exemplo "/Volumes/main/default/industrial_data/industrial_data_serving.csv".
# Não use um caminho local do seu computador Windows.
DATA_SOURCE_TYPE = "table"  # "table" ou "csv"


TEST_SIZE = 0.20
RANDOM_STATE = 42

TARGET_NAME = "big_falha_24h"

In [0]:
# 3. Carregamento e validação inicial dos dados
def load_data(source: str = DATA_SOURCE, source_type: str = DATA_SOURCE_TYPE) -> pd.DataFrame:
    """Carrega dados de uma tabela Spark/Unity Catalog ou de um CSV."""
    if source_type == "table":
        # Disponível dentro de um notebook Databricks com Spark ativo.
        sdf = spark.table(source)
        df = sdf.toPandas()
    elif source_type == "csv":
        df = pd.read_csv(source)
    else:
        raise ValueError("source_type deve ser 'table' ou 'csv'.")

    if df.empty:
        raise ValueError(f"A fonte de dados '{source}' está vazia.")

    print(f"Fonte: {source}")
    print(f"Shape: {df.shape}")
    print("Schema:")
    print(df.dtypes)
    return df


In [0]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder


def data_split(
    df,
    target="big_falha_24h",
    test_size=0.20,
    random_state=42
):
    df = df.copy()

    if target not in df.columns:
        raise ValueError(
            f"A coluna alvo '{target}' não existe no DataFrame."
        )

    # Remove registros sem a variável alvo
    df = df.dropna(subset=[target])

    # Separa features e alvo
    X = df.drop(columns=[target]).copy()
    y = df[target].copy()

    # Converte colunas com nomes relacionados a datas
    date_columns = [
        col for col in X.columns
        if any(term in col.lower() for term in
               ["data", "date", "timestamp", "datetime"])
    ]

    for col in date_columns:
        parsed = pd.to_datetime(X[col], errors="coerce")

        # Só transforma a coluna se ela realmente representar datas
        if parsed.notna().mean() >= 0.5:
            X[f"{col}_ano"] = parsed.dt.year
            X[f"{col}_mes"] = parsed.dt.month
            X[f"{col}_dia"] = parsed.dt.day
            X[f"{col}_dia_semana"] = parsed.dt.dayofweek

            # Remove a data original em formato textual
            X = X.drop(columns=[col])

    # Converte booleanos em 0 e 1
    for col in X.select_dtypes(include=["bool"]).columns:
        X[col] = X[col].astype(int)

    # Trata infinitos
    X = X.replace([np.inf, -np.inf], np.nan)

    # Remove colunas totalmente vazias
    X = X.dropna(axis=1, how="all")

    # Identifica colunas categóricas
    categorical_cols = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    # Divide antes de aprender imputações e codificações
    stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )

    # Codifica categorias sem ajustar usando o conjunto de teste
    if categorical_cols:
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

        X_train[categorical_cols] = encoder.fit_transform(
            X_train[categorical_cols].fillna("__AUSENTE__").astype(str)
        )

        X_test[categorical_cols] = encoder.transform(
            X_test[categorical_cols].fillna("__AUSENTE__").astype(str)
        )

    # Imputação numérica ajustada somente no treino
    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    return X_train, X_test, y_train, y_test


In [0]:
X_train, X_test, y_train, y_test = data_split(load_data(), TARGET_NAME, TEST_SIZE, RANDOM_STATE)
train_df = pd.concat(
    [
        X_train.reset_index(drop=True),
        y_train.reset_index(drop=True).rename(TARGET_NAME),
    ],
    axis=1,
)

test_df = pd.concat(
    [
        X_test.reset_index(drop=True),
        y_test.reset_index(drop=True).rename(TARGET_NAME),
    ],
    axis=1,
)



train_df= spark.createDataFrame(train_df)
test_df= spark.createDataFrame(test_df)

# Salva em Delta
(
    train_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(train_table)
)

(
    test_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(test_table)
)

print(f"Train set salvo: {train_table}")
print(f"Test set salvo: {test_table}")
print(f"Registros de treino: {train_df.count()}")
print(f"Registros de teste: {test_df.count()}")
#


Fonte: datapulse.gold.sensor_events
Shape: (9986, 27)
Schema:
dec_paradas_nao_planejadas             float64
dec_idade_maquina_anos                 float64
dec_temperatura_c                      float64
dec_pressao_bar                        float64
dec_vibracao_motor_mm_s                float64
dec_velocidade_esteira_m_min           float64
dec_umidade_pct                        float64
dec_tamanho_lote                       float64
dec_tempo_setup_min                    float64
dec_taxa_defeitos_pct                  float64
dec_linha_producao_pad_a               float64
dec_linha_producao_pad_b               float64
dec_linha_producao_pad_c               float64
dec_turno_pad_manha                    float64
dec_turno_pad_noite                    float64
dec_turno_pad_tarde                    float64
dec_maquina_m01                        float64
dec_maquina_m02                        float64
dec_maquina_m03                        float64
dec_maquina_m04                        float6